# 03 — Preprocessing & Feature Extraction

Applies the shared cleaning pipeline in `src/preprocess.py` to the raw AG News data, then fits **Bag of Words** and **TF-IDF (unigram + bigram)** vectorizers. Fitted vectorizers and sparse matrices are saved in `models/` for training (notebook 04) and inference.

In [1]:
import sys, os, time
sys.path.insert(0, os.path.abspath('..') if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd())
import pandas as pd, numpy as np, joblib, scipy.sparse as sp
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.preprocessing import LabelEncoder
from src.preprocess import clean_text
os.makedirs('models/features', exist_ok=True); os.makedirs('data/processed', exist_ok=True)

## 1. Load raw data

In [2]:
train = pd.read_csv('ag_news_train.csv'); test = pd.read_csv('ag_news_test.csv')
print(train.shape, test.shape); train.head(3)

(120000, 5) (7600, 5)


,label,title,description,category,text
0,3,Wall St. Bears Claw Back Into the Black (Reuters),"Reuters - Short-sellers, Wall Street's dwindli...",Business,Wall St. Bears Claw Back Into the Black (Reute...
1,3,Carlyle Looks Toward Commercial Aerospace (Reu...,Reuters - Private investment firm Carlyle Grou...,Business,Carlyle Looks Toward Commercial Aerospace (Reu...
2,3,Oil and Economy Cloud Stocks' Outlook (Reuters),Reuters - Soaring crude prices plus worries\ab...,Business,Oil and Economy Cloud Stocks' Outlook (Reuters...


## 2. Clean text
Lowercasing, URL/HTML/special-character removal, stopword removal, Porter stemming; empty or malformed rows are handled (return an empty string). Backslash line-break artefacts in the raw text (e.g. `dwindling\band`) become spaces, so words are no longer glued together as in `train_cleaned.csv`.

In [3]:
t0 = time.time()
for df in (train, test):
    df['clean_text'] = df['text'].apply(clean_text)
print(f'cleaned in {time.time()-t0:.0f}s')
print('empty after cleaning:', (train.clean_text == '').sum(), (test.clean_text == '').sum())
for i in range(2):
    print('RAW  :', train.text[i]); print('CLEAN:', train.clean_text[i], '\n')

cleaned in 4s
empty after cleaning: 0 0
RAW  : Wall St. Bears Claw Back Into the Black (Reuters) Reuters - Short-sellers, Wall Street's dwindling\band of ultra-cynics, are seeing green again.
CLEAN: wall st bear claw black reuter reuter shortsel wall street dwindl band ultracyn see green 

RAW  : Carlyle Looks Toward Commercial Aerospace (Reuters) Reuters - Private investment firm Carlyle Group,\which has a reputation for making well-timed and occasionally\controversial plays in the defense industry, has quietly placed\its bets on another part of the market.
CLEAN: carlyl look commerci aerospac reuter reuter privat invest firm carlyl group reput make welltim occasion controversi play defens industri quietli place bet market 



In [4]:
# drop rows that became empty (none expected), save processed data
train = train[train.clean_text != ''].reset_index(drop=True); test = test[test.clean_text != ''].reset_index(drop=True)
train[['category','clean_text']].to_csv('data/processed/train_processed.csv', index=False)
test[['category','clean_text']].to_csv('data/processed/test_processed.csv', index=False)
print(train.shape, test.shape)

(120000, 6) (7600, 6)


## 3. Label encoding

In [5]:
le = LabelEncoder().fit(train['category'])
y_train, y_test = le.transform(train['category']), le.transform(test['category'])
print(dict(enumerate(le.classes_)))
joblib.dump(le, 'models/label_encoder.joblib'); np.save('models/features/y_train.npy', y_train); np.save('models/features/y_test.npy', y_test)

{0: 'Business', 1: 'Sci/Tech', 2: 'Sports', 3: 'World'}


## 4. Bag of Words

In [6]:
bow = CountVectorizer(max_features=50000, min_df=3)
Xb_train = bow.fit_transform(train.clean_text); Xb_test = bow.transform(test.clean_text)
print(Xb_train.shape, 'nnz/doc', Xb_train.nnz/Xb_train.shape[0])
joblib.dump(bow, 'models/bow_vectorizer.joblib')
sp.save_npz('models/features/bow_train.npz', Xb_train); sp.save_npz('models/features/bow_test.npz', Xb_test)

(120000, 25393) nnz/doc 19.49595


## 5. TF-IDF (unigram + bigram)
The vectorizer is fit on the training set only, then applied to the test set.

In [7]:
tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=100000, min_df=3, sublinear_tf=True)
Xt_train = tfidf.fit_transform(train.clean_text); Xt_test = tfidf.transform(test.clean_text)
print(Xt_train.shape, 'nnz/doc', Xt_train.nnz/Xt_train.shape[0])
joblib.dump(tfidf, 'models/tfidf_vectorizer.joblib')
sp.save_npz('models/features/tfidf_train.npz', Xt_train); sp.save_npz('models/features/tfidf_test.npz', Xt_test)

(120000, 100000) nnz/doc 28.4295


## 6. Sanity check — most distinctive terms per class (mean TF-IDF)

In [8]:
terms = np.array(tfidf.get_feature_names_out())
for k, name in enumerate(le.classes_):
    m = np.asarray(Xt_train[y_train == k].mean(axis=0)).ravel()
    print(f'{name:9s}', ', '.join(terms[m.argsort()[::-1][:10]]))

Business  oil, price, said, reuter, compani, new, stock, profit, percent, sale
Sci/Tech  microsoft, new, softwar, compani, internet, comput, servic, technolog, ap, use
Sports    game, win, ap, team, season, night, victori, cup, leagu, coach
World     iraq, kill, said, ap, presid, minist, reuter, elect, afp, leader


**Word embeddings:** Word2Vec is not used; BoW and TF-IDF are the feature sets carried forward, and TF-IDF is used for the main models.